# 01 — Price Snapshot Explorer

Loads raw `price.snapshot` Avro files from the `market-data` MinIO bucket and visualises intra-day price and volume.

**Requires:** MinIO running and at least one day of data ingested by `mekong-kafka` storage consumer.

In [ ]:
import os
from datetime import date
from dotenv import load_dotenv
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from model.minio_store import MinioStore

load_dotenv()
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)

In [ ]:
store = MinioStore(os.getenv('MINIO_BUCKET', 'market-data'))

# List all available date partitions under price.snapshot/
dates = sorted({
    '/'.join(obj.object_name.split('/')[-4:-1])   # year=.../month=.../day=...
    for obj in store.list_objects(prefix='price.snapshot/')
    if obj.object_name.endswith('.avro')
})
print('Available dates:', dates)

In [ ]:
# Choose which date and asset class to explore
today = date.today()
YEAR   = today.strftime('%Y')
MONTH  = today.strftime('%m')
DAY    = today.strftime('%d')
ASSET  = 'stock'    # 'stock' or 'crypto'

prefix = f'price.snapshot/asset_class={ASSET}/'
date_frag = f'year={YEAR}/month={MONTH}/day={DAY}'

files = [
    obj.object_name
    for obj in store.list_objects(prefix=prefix)
    if date_frag in obj.object_name and obj.object_name.endswith('.avro')
]
print(f'Found {len(files)} Avro files for {ASSET} on {YEAR}-{MONTH}-{DAY}')

In [ ]:
# Read all files into a single DataFrame
frames = []
for key in files:
    rows = store.read_avro(key)
    frames.append(pd.DataFrame(rows))

df = pd.concat(frames, ignore_index=True)
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.sort_values('time').reset_index(drop=True)

print(f'{len(df):,} rows | symbols: {sorted(df.symbol.unique())}')
df.head()

In [ ]:
# Price over time per symbol
fig, ax = plt.subplots()
for symbol, group in df.groupby('symbol'):
    ax.plot(group['time'], group['price'], label=symbol, linewidth=1.2)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_title(f'Intra-day price — {ASSET} — {YEAR}-{MONTH}-{DAY}')
ax.set_xlabel('Time (UTC)')
ax.set_ylabel('Price')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Bid-ask spread per symbol (latest tick)
latest = df.sort_values('time').groupby('symbol').last().reset_index()
latest['spread'] = latest['ask'] - latest['bid']
latest['spread_bps'] = (latest['spread'] / latest['price'] * 10_000).round(2)
latest[['symbol', 'price', 'bid', 'ask', 'spread', 'spread_bps', 'pct_change']].sort_values('pct_change')

In [ ]:
# Volume bar chart — total accumulated volume per symbol
vol = df.groupby('symbol')['volume'].max().sort_values(ascending=False)

fig, ax = plt.subplots()
vol.plot.bar(ax=ax, color='steelblue')
ax.set_title(f'Total volume — {ASSET} — {YEAR}-{MONTH}-{DAY}')
ax.set_ylabel('Volume')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()